# BERT Nested Relation Extraction

In [ ]:
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import itertools
from lark import Lark, Tree, exceptions
import yaml

c:\Users\micha\OneDrive\Documents\Glasgow University\Internship\nested_relation_extraction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.


### Relations and Special Token Initialisation

In [ ]:
one_arg_rels = [
    'increase',
    'decrease',
    'rna_expression',
    'protein_expression',
    'expression',
    'amplification',
    'deletion',
    'mutation',
    'production',
    'bioactivity',
    'phosphorylation',
    'phosphorylated',
    'dephosphorylation',
    'dephosphorylated',
    'ubiquitination',
    'deubiquitination',
    'methylation',
    'methylated',
    'demethylation',
    'demethylated',
]

two_arg_rels = [
    'effect',
    'correlation',
    #'binding'
]

In [4]:
SPECIAL_TOKENS = [
    '[E1]', '[/E1]',
    '[E2]', '[/E2]',
    '[EFFECT]', '[/EFFECT]',
    '[CORRELATION]', '[/CORRELATION]',
    '[INCREASE]', '[/INCREASE]',
    '[DECREASE]', '[/DECREASE]',

    '[RNA_EXPRESSION]', '[/RNA_EXPRESSION]',
    '[PROTEIN_EXPRESSION]', '[/PROTEIN_EXPRESSION]',
    '[EXPRESSION]', '[/EXPRESSION]',
    '[AMPLIFICATION]', '[/AMPLIFICATION]',
    '[DELETION]', '[/DELETION]',
    '[MUTATION]', '[/MUTATION]',
    '[PRODUCTION]', '[/PRODUCTION]',
    '[BIOACTIVITY]', '[/BIOACTIVITY]',
    '[PHOSPHORYLATION]', '[/PHOSPHORYLATION]',
    '[PHOSPHORYLATED]', '[/PHOSPHORYLATED]',
    '[DEPHOSPHORYLATION]', '[/DEPHOSPHORYLATION]',
    '[DEPHOSPHORYLATED]', '[/DEPHOSPHORYLATED]',
    '[UBIQUITINATION]', '[/UBIQUITINATION]',
    '[DEUBIQUITINATION]', '[/DEUBIQUITINATION]',
    '[METHYLATION]', '[/METHYLATION]',
    '[METHYLATED]', '[/METHYLATED]',
    '[DEMETHYLATION]', '[/DEMETHYLATION]',
    '[DEMETHYLATED]', '[/DEMETHYLATED]',
    #'[BINDING]', '[/BINDING]',

    '[CAUSE]', '[/CAUSE]',
    '[THEME]', '[/THEME]',
    '[VARIABLE]', '[/VARIABLE]',
    '[GENE_PROTEIN]', '[/GENE_PROTEIN]',
    '[BIOMOLECULE]', '[/BIOMOLECULE]',
    #'[PARTNER1]', '[/PARTNER2]',

]

### Tokeniser 

In [ ]:
MODEL_NAME = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_tokens(SPECIAL_TOKENS, True)

c:\Users\micha\OneDrive\Documents\Glasgow University\Internship\nested_relation_extraction\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\micha\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 78

Embedding(29054, 768, padding_idx=0)

## BERT Model Training

In [ ]:
all_relation_labels = one_arg_rels + two_arg_rels + ["none"]
label2id = {label: i for i, label in enumerate(all_relation_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label2id)
print(num_labels, label2id)

In [ ]:
def tokenize_and_filter(batch, tokenizer, label2id, max_length=512):
    encodings = tokenizer(
        batch["text"],
        truncation=False,
    )

    keep_indices = [
        i for i, ids in enumerate(encodings["input_ids"])
        if len(ids) <= max_length
    ]

    filtered = {
        key: [values[i] for i in keep_indices]
        for key, values in encodings.items()
    }
    filtered["labels"] = [label2id[batch["label"][i]] for i in keep_indices]

    return filtered


def build_relation_dataset(examples, tokenizer, label2id, max_length=512):
    dataset = Dataset.from_dict({
        "text": [ex["text"] for ex in examples],
        "label": [ex["label"] for ex in examples],
    })

    return dataset.map(
        lambda batch: tokenize_and_filter(batch, tokenizer, label2id, max_length),
        batched=True,
        remove_columns=["text", "label"],
    )

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
classifier = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)
classifier.resize_token_embeddings(len(tokenizer))

In [ ]:
with open('train_bert_training_examples_small.json') as f:
    train_examples = json.load(f)

with open('val_bert_training_examples_small.json') as f:
    val_examples = json.load(f)

train_dataset = build_relation_dataset(train_examples, tokenizer, label2id)
val_dataset = build_relation_dataset(val_examples, tokenizer, label2id)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": f1,
        "macro_precision": precision,
        "macro_recall": recall,
    }

training_args = TrainingArguments(
    output_dir="./phase4_minimal",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True
)

trainer = Trainer(
    model=classifier,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.model.eval()
eval_device = trainer.args.device
sample = val_examples[:10]
with torch.no_grad():
    for ex in sample:
        encoded = tokenizer(ex["text"], truncation=True, max_length=256, return_tensors="pt").to(eval_device)
        logits = trainer.model(encoded["input_ids"], encoded["attention_mask"])["logits"]
        pred_label = id2label[logits.argmax(dim=-1).item()]
        print(f"gold={ex['label']:<15} pred={pred_label:<15} text={ex['text'][:80]}...")

In [ ]:
trainer.save_model("./phase4_minimal/final_model")
tokenizer.save_pretrained("./phase4_minimal/final_model")

In [ ]:
predictions = trainer.predict(val_dataset)
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

report_dict = classification_report(
    y_true,
    y_pred,
    labels=list(range(num_labels)),
    target_names=all_relation_labels,
    digits=4,
    zero_division=0,
    output_dict=True,
)

with open("bert_classification_report.json", "w") as f:
    json.dump(report_dict, f, indent=2)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)), normalize="true")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=all_relation_labels
)

fig, ax = plt.subplots(figsize=(12, 12))
disp.plot(
    cmap="Blues",
    ax=ax,
    xticks_rotation=90,
    colorbar=False
)

plt.tight_layout()
plt.savefig('cm_v2bert_base_cased.png')
plt.show()

## Multi-Pass BERT Inference Loop

In [ ]:
class Node:
    def __init__(self, kind, label):
        self.kind = kind
        self.label = label
        self.children = []
        self.level = 0
        self.parent_arg = None
        self.discovered = False

    def flat_text(self):
        if self.kind == 'entity':
            return self.label
        else:
            inner = " ".join(f"[{a.upper()}]{c.flat_text()}[/{a.upper()}]" for a, c in self.children)
            tag = self.label.upper()
            return f"[{tag}]{inner}[/{tag}]"

    def output_text(self):
        if self.kind == 'entity':
            return self.label
        else:
            inner = ", ". join(f"{a}={c.output_text()}" for a, c in self.children)
            return f"{self.label}({inner})"

def get_nodes_by_level(nodes):
    level_dict = {}
    for node in nodes:
        if node.level not in level_dict:
            level_dict[node.level] = []
        level_dict[node.level].append(node)
    return level_dict

def render_context_segment(node, tag=None):
    text = node.flat_text()
    return f"[{tag}]{text}[/{tag}]" if tag else text

def parse_relation(rel):
    if rel['type'] == 'entity':
        return Node('entity', rel['entity'])
    else:
        node = Node('relation', rel['label'])
        for arg_name, child_json in rel['arguments'].items():
            child = parse_relation(child_json)
            node.children.append((arg_name, child))
            child.parent_arg = (node, arg_name)
        return node

def compute_levels(node):
    if node.kind == 'entity':
        node.level = 0
    else:
        node.level = 1 + max(compute_levels(c) for _, c in node.children)
    return node.level

def collect_all(node, acc=None):
    if acc is None:
        acc = []
    acc.append(node)
    for _, c in node.children:
        collect_all(c, acc)
    return acc


In [ ]:
example = {
    "relation_text": "effect(cause=increase(variable=expression(gene_protein=TP53)), theme=methylation(gene_protein=PTEN))",
    "sentence": "We studied the effect of an increase in TP53 expression on PTEN methylation.",
    "entities": ['TP53', 'PTEN'],
    "relation": {
        "type": "relation",
        "label": "effect",
        "arguments": {
            "cause" : {
                "type": "relation",
                "label": "increase",
                "arguments": {
                    "variable" : {
                          "type": "relation",
                      "label": "expression",
                      "arguments": {
                          "gene_protein": {
                                "type": "entity",
                                "entity": "TP53"
                          }
                      }
                    }

                }
            },
            "theme": {
                "type": "relation",
                "label": "methylation",
                "arguments": {
                    "gene_protein": {
                        "type": "entity",
                        "entity": "PTEN"
                    }
                }
            }
        }
    }
}

In [ ]:
model_path = "./phase4_minimal/final_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def predict_relation(text, model, tokenizer, id2label, device, max_length=512):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
    ).to(device)

    seq_len = inputs["input_ids"].shape[1]
    if seq_len > max_length:
        return 'none'

    with torch.no_grad():
        logits = model(**inputs).logits

    pred_id = torch.argmax(logits, dim=1).item()
    return id2label[pred_id]


def create_entity_node(entity):
    return Node('entity', entity)

In [ ]:
single_arg_names = {
    'increase': 'variable',
    'decrease': 'variable',
    'rna_expression': 'gene_protein',
    'protein_expression': 'gene_protein',
    'expression': 'gene_protein',
    'amplification': 'gene_protein',
    'deletion': 'gene_protein',
    'mutation': 'gene_protein',
    'production': 'biomolecule',
    'bioactivity': 'biomolecule',
    'phosphorylation': 'gene_protein',
    'phosphorylated': 'gene_protein',
    'dephosphorylation': 'gene_protein',
    'dephosphorylated': 'gene_protein',
    'ubiquitination': 'gene_protein',
    'deubiquitination': 'gene_protein',
    'methylation': 'gene_protein',
    'methylated': 'gene_protein',
    'demethylation': 'gene_protein',
    'demethylated': 'gene_protein',
}

two_arg_names = {
    'effect': ('cause', 'theme'),
    'correlation': ('cause', 'theme')
}

In [ ]:
def bert_multipass_inference(sentence, entities, model, tokenizer, id2label, device, verbose=False):
    entity_nodes = [create_entity_node(e) for e in entities]
    discovered = list(entity_nodes)
    pool = list(entity_nodes)
    counter = 1
    
    while True:
        if verbose:
            print("="*40)
            print(f"\nRound {counter}")
            print("="*40)
            
        new_this_round = []
        discovered_this_round = []

        for candidate in pool:
            if candidate.discovered:
                continue
                
            ctx_parts = [render_context_segment(n, 'E1' if n is candidate else None) for n in discovered]
            new_text = sentence + " [SEP]" + "[SEP]".join(ctx_parts)

            label = predict_relation(new_text, model, tokenizer, id2label, device)
            if verbose:
                print(f"\n{new_text}")
                print(label)
            
            if label != 'none' and single_arg_names.get(label, None):
                new_node = Node('relation', label)
                arg_name = single_arg_names[label]
                new_node.children.append((arg_name, candidate))
                candidate.parent_arg = (new_node, arg_name)

                discovered_this_round.append(candidate)
                new_this_round.append(new_node)


        for c1, c2 in itertools.permutations(pool, 2):
            if c1.discovered or c2.discovered:
                continue

            ctx_marks = {c1: 'E1', c2: 'E2'}
            ctx_parts = [render_context_segment(n, ctx_marks.get(n)) for n in discovered]
            new_text = sentence + " [SEP]" + "[SEP]".join(ctx_parts)

            label = predict_relation(new_text, model, tokenizer, id2label, device)
            if verbose:
                print(f"\n{new_text}")
                print(label)

            if label != 'none' and two_arg_names.get(label, None):
                new_node = Node('relation', label)
                args = two_arg_names[label]
                new_node.children.append((args[0], c1))
                c1.parent_arg = (new_node, args[0])
                new_node.children.append((args[1], c2))
                c2.parent_arg = (new_node, args[1])

                if c1 not in discovered_this_round:
                    discovered_this_round.append(c1)
                if c2 not in discovered_this_round:
                    discovered_this_round.append(c2)
                new_this_round.append(new_node)
                
            

        for c in discovered_this_round:
            c.discovered = True
            
        if not new_this_round:
            break
        for parent in new_this_round:
            discovered.append(parent)
            pool.append(parent)

        counter += 1

    output = [node.output_text() for node in discovered if node.kind == 'relation']
    return output

In [ ]:
node = bert_multipass_inference(example['sentence'], example['entities'], model, tokenizer, id2label, device)
print(node.flat_text())
print(node.output_text())

In [ ]:
sentence = "In the present study, the dephosphorylation of matrix metalloproteinase‑9 (MMP9) was found to constitute a pivotal upstream event that directly modulates the RNA expression of KRAS, such that the removal of phosphate groups from MMP9 precipitates a measurable alteration in KRAS mRNA abundance, thereby establishing a causal link between MMP9 dephosphorylation and the transcriptional regulation of KRAS."
entities = [
    'MMP9',
    'KRAS'
]
node = bert_multipass_inference(sentence, entities, model, tokenizer, id2label, device, verbose=True)

In [ ]:
print(node.flat_text())
print(node.output_text())

### Metrics (Borrowed from T5 pipeline)

In [ ]:
grammar = r"""
  start: predicate

  predicate: NAME "(" [arg ("," arg)*] ")"

  arg: NAME "=" value

  value: predicate
     | STRING

  NAME: /[a-zA-Z_][a-zA-Z0-9_]*/
  STRING: /(?s)\[QUOTE\].*?\[QUOTE\]/

  %import common.WS
  %ignore WS
"""

parser = Lark(grammar, start="start", parser="earley")

def parse_tree(text):
    try:
        tree = parser.parse(text)
        return tree, None
    except exceptions.UnexpectedCharacters as e:
        return None, f"unexpected_char at pos {e.pos_in_stream}: {e}"
    except exceptions.UnexpectedEOF as e:
        return None, "truncated_input"
    except exceptions.UnexpectedToken as e:
        return None, f"unexpected_token at pos {e.pos_in_stream}: {e}"
    except exceptions.LarkError as e:
        return None, f"parse_error: {e}"

def parse_error_category(err):
    if err is None:
        return "success"
    if err == "truncated_input":
        return "truncated_input"
    if err.startswith("unexpected_char"):
        return "unexpected_char"
    if err.startswith("unexpected_token"):
        return "unexpected_token"
    return "other_parse_error"

In [ ]:
def load_terms(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    terms = {category: set(entities) for category, entities in raw.items()}
    terms['biomolecule'] = terms['chemical'].union(terms['gene'])
    terms['gene_protein'] = terms['gene']
    return terms


def load_relation_schema(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def strip_quote_markers(string_token_text):
    return string_token_text[len("[QUOTE]"):-len("[QUOTE]")]

In [ ]:
def validate_predicate(predicate_tree, allowed_types, relation_schema, terms):
    label = predicate_tree.children[0].value
    arg_nodes = [c for c in predicate_tree.children[1:] if c is not None]

    if label not in relation_schema:
        return False, f"unknownn_relation_label: '{label}' is not defined in relation_schema"

    if allowed_types is not None and label not in allowed_types:
        return False, (
            f"disallowed_relation_type: '{label}' is not permitted in this position "
            f"(allowed: {allowed_types})"
        )

    definition = relation_schema[label]
    required_args = set(definition["args"])

    present = {}
    for arg_node in arg_nodes:
        arg_name = arg_node.children[0].value
        arg_tree = arg_node.children[1]
        present[arg_name] = arg_tree

    present_names = set(present.keys())

    missing = required_args - present_names
    if missing:
        return False, f"missing_args: '{label}' is missing required arg(s) {sorted(missing)}"

    unexpected = present_names - required_args
    if unexpected:
        return False, f"unexpected_args: '{label}' has unrecognized arg(s) {sorted(unexpected)}"

    for arg_name in required_args:
        arg_tree = present[arg_name]
        arg_types = definition.get("arg_types", {}).get(arg_name, [arg_name])
        ok, err = validate_arg(arg_tree, arg_types, relation_schema, terms)
        if not ok:
            return False, f"in {label}.{arg_name}: {err}"

    return True, None


def validate_arg(arg_tree, allowed_types, relation_schema, terms):
    inner = arg_tree.children[0]

    if isinstance(inner, Tree) and inner.data == "predicate":
        return validate_predicate(inner, allowed_types, relation_schema, terms)

    entity_str = strip_quote_markers(str(inner))
    entity_categories = [t for t in allowed_types if t in terms]

    for cat in entity_categories:
        if entity_str in terms[cat]:  # O(1): terms[cat] is a set
            return True, None

    if not entity_categories:
        return False, (
            f"unexpected_entity_position: got a bare entity '{entity_str}' but none of the "
            f"allowed types here are entity categories (allowed_types={allowed_types})"
        )
    return False, (
        f"unknown_entity: '{entity_str}' not found in any allowed category {entity_categories}"
    )

In [ ]:
def decompose(predicate_tree):
    def _decompose(node):
        pred_name = node.children[0].value
        rels = set()
        children_repr = []

        for arg_node in node.children[1:]:
            arg_name = arg_node.children[0].value
            inner = arg_node.children[1].children[0]

            if isinstance(inner, Tree) and inner.data == "predicate":
                child_id, child_rels = _decompose(inner)
                rels |= child_rels
                children_repr.append((arg_name, child_id))
            else:
                entity = strip_quote_markers(str(inner))
                children_repr.append((arg_name, entity))

        current = (hash((pred_name, tuple(children_repr))), pred_name)

        for arg_name, child in children_repr:
            rels.add((current, arg_name, child))

        return current, rels

    _, binary_rels = _decompose(predicate_tree)
    return binary_rels


def get_bin_rels(valid_preds, valid_targets):
    all_pred_bin_rels = []
    all_target_bin_rels = []

    for p, t in zip(valid_preds, valid_targets):
        if isinstance(p, list):
            pred_bin_rels = set().union(*(decompose(pred) for pred in p))
        else:
            pred_bin_rels = decompose(p)
            
        target_bin_rels = decompose(t)

        all_pred_bin_rels.append(pred_bin_rels)
        all_target_bin_rels.append(target_bin_rels)

    return all_pred_bin_rels, all_target_bin_rels


def get_parse_fail_FN(invalid_targets):
    FN = 0
    for t in invalid_targets:
        bin_rels = decompose(t)
        FN += len(bin_rels)

    return FN


def binary_decomposition_metrics(pred_bin_rels, target_bin_rels, parse_fail_FN):
    TP = 0
    FP = 0
    FN = parse_fail_FN

    for p_bin_rels, t_bin_rels in zip(pred_bin_rels, target_bin_rels):
        TP += len(t_bin_rels & p_bin_rels)
        FP += len(p_bin_rels.difference(t_bin_rels))
        FN += len(t_bin_rels.difference(p_bin_rels))

    precision = TP / (TP+FP) if (TP+FP) else 0
    recall = TP / (TP+FN) if (TP+FN) else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        "bin_decomp_precision" : precision,
        "bin_decomp_recall" : recall,
        "bin_decomp_f1_score" : f1_score,
    }

### BERT Multi-Pass Metric Evaluation

In [ ]:
def evaluate_bert(preds, targets, relation_schema, terms, root_labels):
    n = 0
    valid_parse_count = 0
    valid_schema_count = 0

    valid_preds = []
    valid_targets = []
    invalid_parse_targets = []

    for p, t in zip(preds, targets):
        parsed_target, _ = parse_tree(t)
        parsed_target = parsed_target.children[0]

        valid_parses = []
        n += len(p)
        for pred in p:
            parsed_pred, err = parse_tree(pred)
            if err is None:
                parsed_pred = parsed_pred.children[0]
                valid_parse_count += 1
    
                #Schema check
                ok, err = validate_predicate(parsed_pred, root_labels, relation_schema, terms)
                if ok:
                    valid_schema_count += 1

                valid_parses.append(parsed_pred)

        if valid_parses:
            valid_preds.append(valid_parses)
            valid_targets.append(parsed_target)
        else:
            invalid_parse_targets.append(parsed_target)

    parse_rate = valid_parse_count / n
    schema_rate = valid_schema_count / n

    pred_bin_rels, target_bin_rels = get_bin_rels(valid_preds, valid_targets)
    parse_fail_FN = get_parse_fail_FN(invalid_parse_targets)
    bin_decomp_metrics = binary_decomposition_metrics(pred_bin_rels, target_bin_rels, parse_fail_FN)

    return {
        "parse_rate": parse_rate,
        "schema_rate": schema_rate,
        **bin_decomp_metrics,

    }        

In [ ]:
with open('val_rels.json') as f:
    val_rels = json.load(f)

targets = [rel['relation_text'] for rel in val_rels]
preds = [
    bert_multipass_inference(example['sentence'], example['entities'], model, tokenizer, id2label, device)
    for example in val_rels
]

BERT_ROOT_LABELS = one_arg_rels + two_arg_rels
SCHEMA_PATH = "../relation_schema.yml"
TERMS_PATH = "../resources/terms.json"

relation_schema = load_relation_schema(SCHEMA_PATH)
terms = load_terms(TERMS_PATH)
evaluate_bert(preds, targets, relation_schema, terms, root_labels=BERT_ROOT_LABELS)